# aws

> SageMaker training jobs from a notebook or a script, on CPUs, NVIDIA GPUs or Trainium, with one container image that trains and serves; tested here without an AWS account against floci, a local AWS emulator.

In [ ]:
#| default_exp aws

In [ ]:
#| export
import io, json, os, re, shutil, subprocess, sys, tarfile, tempfile, time, urllib.request
from pathlib import Path

import sysone
from sysone.cloud import (SageMaker, CPU, GPU, TPU, Neuron, Job, LocalPath, S3Data, machine, stage, submit, poll, cancel, collect,
                          pack, build_wheel, unpack_outputs, save_job, DONE)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from sysone.cloud import remote, jobs
import tarfile
from sysone.inference import Decider

A SageMaker training job runs a container image: SageMaker copies the job's inputs from S3 into the container (`/opt/ml/input/data/<channel>`), runs its `train` command, uploads what it leaves in `/opt/ml/model` as `model.tar.gz`, and sends its output to CloudWatch Logs. An endpoint runs the same kind of image with `serve`, which must answer `GET /ping` and `POST /invocations` on port 8080. sysone's image does both:

- `train` unpacks the sysone job that arrives as the `code` channel and runs its [runner](40_cloud.ipynb#the-jobs-files), as on Colab or Kaggle; the export goes to `/opt/ml/model`, the job's other outputs to `/opt/ml/output/data`;
- `serve` loads `/opt/ml/model` with `Decider.load` and runs sysone's server, which has SageMaker's routes (see [inference](06_inference.ipynb#serving)).

The calls are plain boto3 (`pip install 'sysone[aws]'`), so the SageMaker Python SDK is not needed to run jobs; [below](#with-the-sagemaker-python-sdk) is how the SDK's `ModelTrainer` uses the same image. Every client goes to `SageMaker(endpoint_url=...)` when it is set, which is how this notebook runs against [floci](https://floci.io), an AWS emulator in a Docker container: S3, IAM, CloudWatch Logs, CloudFormation, and, since September 2026, SageMaker's training jobs and endpoints, run as real containers on this machine's Docker. Against an emulator, a missing bucket is made and a dummy role is used; against AWS, both are named (`SageMaker(role=..., bucket=...)`, or `SYSONE_SAGEMAKER_ROLE` and `SYSONE_SAGEMAKER_BUCKET`).

::: {.callout-note title="Finding: floci runs SageMaker's training jobs and endpoints for real, with two gaps"}
On floci's nightly of 2026-09-30, a training job runs the image (a local tag is enough) in a container with SageMaker's `/opt/ml` layout, its channels copied in, its output in CloudWatch Logs and `/opt/ml/model` uploaded as `model.tar.gz`; an endpoint runs `serve` and answers `InvokeEndpoint`. Two things differ from SageMaker: a training job's `Environment` does not reach the container, and `/opt/ml/output/data` is not uploaded. So a sysone job carries its settings in its own `job.json`, as on every backend, and on an emulator it puts its outputs in the model too (`outputs_in_model`), where `fetch` finds them.
:::

In [ ]:
#| export
FLOCI_IMAGE = "floci/floci:nightly-09302026"
EMULATOR_ROLE = "arn:aws:iam::000000000000:role/sysone-sagemaker"

def emulated(b:SageMaker) -> bool:
    "Whether the backend talks to an emulator (an `endpoint_url`) rather than AWS"
    return b.endpoint_url is not None

def aws_client(b:SageMaker, service:str):
    "A boto3 client for `service` in the backend's region, at its endpoint when it has one (dummy keys for an emulator)"
    import boto3
    kw = {"region_name": b.region or os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-east-1"}
    if b.endpoint_url:
        kw["endpoint_url"] = b.endpoint_url
        if not os.environ.get("AWS_ACCESS_KEY_ID"): kw |= {"aws_access_key_id": "test", "aws_secret_access_key": "test"}
    return boto3.client(service, **kw)

def role_arn(b:SageMaker) -> str:
    "The execution role jobs and endpoints run as"
    r = b.role or os.environ.get("SYSONE_SAGEMAKER_ROLE")
    if r: return r
    if emulated(b): return EMULATOR_ROLE
    raise ValueError("name the SageMaker execution role: SageMaker(role='arn:aws:iam::ACCOUNT:role/NAME') or SYSONE_SAGEMAKER_ROLE")

def bucket(b:SageMaker) -> str:
    "The S3 bucket for jobs' code, data and outputs (made on an emulator when missing)"
    name = b.bucket or os.environ.get("SYSONE_SAGEMAKER_BUCKET") or ("sysone" if emulated(b) else None)
    if name is None: raise ValueError("name the S3 bucket for jobs: SageMaker(bucket=...) or SYSONE_SAGEMAKER_BUCKET")
    s3 = aws_client(b, "s3")
    try: s3.head_bucket(Bucket=name)
    except Exception:
        if not emulated(b): raise ValueError(f"the bucket {name!r} does not exist, or this identity cannot reach it") from None
        s3.create_bucket(Bucket=name)
    return name

`floci_up` starts the emulator in Docker (with the Docker socket, which it needs to run training and endpoint containers) and waits for it; `floci_down` removes it. The image is a dated nightly, since SageMaker support came after floci's last release.

In [ ]:
#| export
def floci_up(port:int=4566, image:str=FLOCI_IMAGE, name:str="sysone-floci", timeout:float=60) -> str:
    "Start floci, a local AWS emulator, in Docker (or find it running); returns its endpoint URL"
    url = f"http://localhost:{port}"
    up = lambda: subprocess.run(["docker", "ps", "-q", "-f", f"name=^{name}$"], capture_output=True, text=True).stdout.strip()
    if not up():
        subprocess.run(["docker", "rm", "-f", name], capture_output=True)
        subprocess.run(["docker", "run", "-d", "--name", name, "-p", f"127.0.0.1:{port}:4566", "-v", "/var/run/docker.sock:/var/run/docker.sock", image],
                       check=True, capture_output=True)
    t0 = time.time()
    while True:
        try: urllib.request.urlopen(url, timeout=2); return url
        except urllib.error.HTTPError: return url            # it answers, with an error for a bare GET
        except Exception:
            if time.time() - t0 > timeout: raise RuntimeError(f"floci did not answer at {url}; docker logs {name}")
            time.sleep(0.5)

def floci_down(name:str="sysone-floci"):
    "Remove the floci container `floci_up` started"
    subprocess.run(["docker", "rm", "-f", name], capture_output=True)

## Instances

SageMaker names machines by instance type. A `GPU` maps to the instance with that many of that GPU; Trainium and Inferentia are `Neuron("trn1.2xlarge")` and the like. `SageMaker(local=True)` is the SDK's local mode, whose instance types are `local` and `local_gpu`.

In [ ]:
#| export
SAGEMAKER_GPUS = {("T4", 1): "ml.g4dn.xlarge", ("T4", 4): "ml.g4dn.12xlarge", ("A10G", 1): "ml.g5.xlarge", ("A10G", 4): "ml.g5.12xlarge",
                  ("L4", 1): "ml.g6.xlarge", ("L4", 4): "ml.g6.12xlarge", ("A100", 8): "ml.p4d.24xlarge", ("H100", 8): "ml.p5.48xlarge"}

@machine.register
def machine_sagemaker_cpu(b:SageMaker, hw:CPU) -> str: return "local" if b.local else ("ml.r5.xlarge" if hw.high_mem else "ml.m5.xlarge")

@machine.register
def machine_sagemaker_gpu(b:SageMaker, hw:GPU) -> str:
    if b.local: return "local_gpu"
    if (hw.name, hw.count) not in SAGEMAKER_GPUS:
        raise ValueError(f"no SageMaker instance has {hw.count} {hw.name}; there are " + ", ".join(f"{n}x{c}" for n, c in SAGEMAKER_GPUS))
    return SAGEMAKER_GPUS[(hw.name, hw.count)]

@machine.register
def machine_sagemaker_neuron(b:SageMaker, hw:Neuron) -> str:
    if b.local: raise ValueError("the SDK's local mode has no Neuron devices")
    return f"ml.{hw.kind}"

In [ ]:
test_eq([machine(SageMaker(), h) for h in (CPU(), GPU("T4"), GPU("L4"), GPU("A100", 8), Neuron("trn1.2xlarge"))],
        ["ml.m5.xlarge", "ml.g4dn.xlarge", "ml.g6.xlarge", "ml.p4d.24xlarge", "ml.trn1.2xlarge"])
test_eq((machine(SageMaker(local=True), CPU()), machine(SageMaker(local=True), GPU("T4"))), ("local", "local_gpu"))
test_fail(lambda: machine(SageMaker(), GPU("H100", 2)), contains="no SageMaker instance")
test_fail(lambda: machine(SageMaker(), TPU()), contains="sagemaker has no tpu")

## The image

`dockerfile` writes the image's recipe: a CPU image starts from Python's slim image with PyTorch's CPU wheels; a GPU image starts from PyTorch's CUDA image. Both install the wheel of this sysone checkout (or the release, from PyPI), sysone's server, and nbdev's notebook runner, and add the two commands. `build_image` builds it with Docker; for AWS, build it for `linux/amd64` and push it to ECR (an emulator runs a local tag as it is).

In [ ]:
#| export
TRAIN = r'''#!/usr/bin/env python3
# SageMaker's training command for a sysone job: the job's folder arrives as the `code` channel
import glob, json, os, shutil, subprocess, sys, tarfile
CODE, MODEL = "/opt/ml/code", "/opt/ml/model"
src = sorted(glob.glob("/opt/ml/input/data/code/*.tar.gz"))
if not src: sys.exit("no sysone job in the code channel")
os.makedirs(CODE, exist_ok=True)
with tarfile.open(src[0]) as t: t.extractall(CODE, filter="data")
rc = subprocess.run([sys.executable, os.path.join(CODE, "sysone_job.py")]).returncode
outputs, spec = os.path.join(CODE, "outputs"), json.load(open(os.path.join(CODE, "job.json")))
os.makedirs(MODEL, exist_ok=True)
if os.path.isfile(os.path.join(outputs, "export", "sysone.json")): shutil.copytree(os.path.join(outputs, "export"), MODEL, dirs_exist_ok=True)
tb = os.path.join(CODE, "outputs.tar.gz")
if os.path.isfile(tb):
    os.makedirs("/opt/ml/output/data", exist_ok=True)
    shutil.copy(tb, "/opt/ml/output/data/outputs.tar.gz")
    if spec.get("outputs_in_model"): shutil.copy(tb, os.path.join(MODEL, "sysone_outputs.tar.gz"))   # an emulator uploads the model only
if rc != 0:
    st = os.path.join(outputs, "status.json")
    with open("/opt/ml/output/failure", "w") as f: f.write(open(st).read() if os.path.isfile(st) else f"the job exited with {rc}")
sys.exit(rc)
'''

SERVE = r'''#!/usr/bin/env python3
# SageMaker's serving command: sysone's server over the model in /opt/ml/model, on port 8080
import os, uvicorn
from sysone.inference import Decider, serve_app
model = os.environ.get("SYSONE_MODEL", "/opt/ml/model")
uvicorn.run(serve_app(Decider.load(model, device=os.environ.get("SYSONE_DEVICE") or None)), host="0.0.0.0",
            port=int(os.environ.get("SAGEMAKER_BIND_TO_PORT", "8080")))
'''

def dockerfile(gpu:bool=False, base:str|None=None, wheel:bool=True) -> str:
    "The recipe of sysone's SageMaker image: CPU (Python's slim image, PyTorch's CPU wheels) or GPU (PyTorch's CUDA image)"
    base = base or ("pytorch/pytorch:2.14.1-cuda12.6-cudnn9-runtime" if gpu else "python:3.12-slim")
    torch = "" if gpu else 'RUN pip install --index-url https://download.pytorch.org/whl/cpu "torch>=2.8"\n'
    pkg = "/opt/sysone/wheels/*.whl" if wheel else f'"sysone=={sysone.__version__}"'
    return (f"# sysone's SageMaker image: `train` runs a sysone job, `serve` answers /ping and /invocations on port 8080\n"
            f"FROM {base}\nENV PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1 PIP_DISABLE_PIP_VERSION_CHECK=1\n{torch}"
            + ("COPY wheels/ /opt/sysone/wheels/\n" if wheel else "")
            + f'RUN pip install {pkg} "fastapi>=0.110" "uvicorn>=0.30" execnb\n'
            "COPY train serve /usr/local/bin/\nRUN chmod +x /usr/local/bin/train /usr/local/bin/serve\nWORKDIR /opt/ml\nEXPOSE 8080\n"
            'CMD ["train"]\n')     # SageMaker runs `train` or `serve`; the SDK's local mode runs the default command

def image_tag(gpu:bool=False) -> str: return f"sysone:{sysone.__version__}-{'gpu' if gpu else 'cpu'}"

def image_context(dest=None, gpu:bool=False, base:str|None=None) -> Path:
    "A folder Docker builds sysone's image from: the recipe, the two commands and the wheel"
    ctx = Path(dest or tempfile.mkdtemp(prefix="sysone-image-")); ctx.mkdir(parents=True, exist_ok=True)
    whl = build_wheel(ctx / "wheels")
    (ctx / "Dockerfile").write_text(dockerfile(gpu, base, wheel=whl is not None))
    (ctx / "train").write_text(TRAIN); (ctx / "serve").write_text(SERVE)
    return ctx

def build_image(tag:str|None=None, gpu:bool=False, platform:str|None=None, base:str|None=None, log=None) -> str:
    "Build sysone's image with Docker; returns its tag (`sysone:VERSION-cpu` by default)"
    tag = tag or image_tag(gpu)
    ctx = image_context(gpu=gpu, base=base)
    cmd = ["docker", "build", "-t", tag] + (["--platform", platform] if platform else []) + [str(ctx)]
    with open(log or ctx / "build.log", "w") as f: r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0: raise RuntimeError(f"docker build failed; see {log or ctx / 'build.log'}")
    return tag

In [ ]:
compile(TRAIN, "train", "exec"); compile(SERVE, "serve", "exec")
ctx = image_context()
test_eq(sorted(p.name for p in ctx.iterdir()), ["Dockerfile", "serve", "train", "wheels"])
print((ctx / "Dockerfile").read_text())

## Data and jobs

Data reaches a training job as a channel: local data is uploaded under the job's prefix when it is submitted, and S3 data is read where it is. The job finds it in `SYSONE_DATA` as usual: the channel's folder, or the file in it when one file was given. The Hub token cannot reach the container (an endpoint's or a job's environment is readable by anyone who can describe it), so, as on Kaggle, `job.fetch()` pushes the export from here.

In [ ]:
#| export
def _channel(var:str) -> str: return var.lower().removeprefix("sysone_") or "data"

@stage.register
def stage_sagemaker_local(d:LocalPath, b:SageMaker, var:str) -> dict:
    p, ch = Path(d.path), _channel(var)
    return {"upload": {ch: d.path}, "find": {var: f"/opt/ml/input/data/{ch}" + (f"/{p.name}" if p.is_file() else "")}}

@stage.register
def stage_sagemaker_s3(d:S3Data, b:SageMaker, var:str) -> dict:
    ch = _channel(var)
    return {"channels": {ch: d.uri}, "find": {var: f"/opt/ml/input/data/{ch}"}}

def s3_put(s3, bucket:str, prefix:str, path:Path) -> str:
    "Upload a file or a folder under `prefix`; returns the S3 prefix"
    path = Path(path)
    files = [path] if path.is_file() else [f for f in sorted(path.rglob("*")) if f.is_file()]
    for f in files:
        key = f"{prefix}/{f.name if path.is_file() else f.relative_to(path).as_posix()}"
        s3.upload_file(str(f), bucket, key)
    return f"s3://{bucket}/{prefix}/"

def image_for(b:SageMaker, machine:str) -> str:
    "The image a job runs: the backend's, SYSONE_SAGEMAKER_IMAGE, or sysone's GPU or CPU tag by the instance"
    gpu = machine == "local_gpu" or bool(re.match(r"ml\.(g\d|p\d)", machine))
    return b.image or os.environ.get("SYSONE_SAGEMAKER_IMAGE") or image_tag(gpu)

def training_job_request(b:SageMaker, job:Job, spec:dict, channels:dict, bucket:str, prefix:str) -> dict:
    "The CreateTrainingJob request for a job"
    return {"TrainingJobName": job.id[:63], "RoleArn": role_arn(b),
            "AlgorithmSpecification": {"TrainingImage": image_for(b, job.machine), "TrainingInputMode": "File"},
            "InputDataConfig": [{"ChannelName": ch, "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": uri,
                                                                                    "S3DataDistributionType": "FullyReplicated"}}}
                                for ch, uri in channels.items()],
            "OutputDataConfig": {"S3OutputPath": f"s3://{bucket}/{prefix}/output"},
            "ResourceConfig": {"InstanceType": job.machine, "InstanceCount": 1, "VolumeSizeInGB": 50},
            "StoppingCondition": {"MaxRuntimeInSeconds": int(spec.get("max_seconds") or 24 * 3600)},
            "Tags": [{"Key": "sysone-job", "Value": job.id}]}

In [ ]:
test_eq(stage(LocalPath("/d/train.jsonl"), SageMaker(), "SYSONE_DATA")["find"]["SYSONE_DATA"], "/opt/ml/input/data/data")   # not a file here
test_eq(stage(S3Data("s3://b/emotions/"), SageMaker(), "SYSONE_DATA_VALID"), {"channels": {"data_valid": "s3://b/emotions/"}, "find": {"SYSONE_DATA_VALID": "/opt/ml/input/data/data_valid"}})
fake = Job("sagemaker-20261001-120000-abcd", "sagemaker", "ml.g4dn.xlarge", "train.py")
req = training_job_request(SageMaker(endpoint_url="http://localhost:4566"), fake, {"max_seconds": 3600}, {"code": "s3://sysone/p/code/"}, "sysone", "p")
test_eq((req["AlgorithmSpecification"]["TrainingImage"], req["ResourceConfig"]["InstanceType"], req["StoppingCondition"]["MaxRuntimeInSeconds"], req["RoleArn"]),
        (image_tag(gpu=True), "ml.g4dn.xlarge", 3600, EMULATOR_ROLE))
test_fail(lambda: role_arn(SageMaker()), contains="execution role")

`submit` uploads the job's folder as the `code` channel and the local data as channels of their own, and creates the training job. `poll` describes it and reads its CloudWatch log streams from where the last poll stopped (the job's progress lines among them); when it ends, it downloads `model.tar.gz` and the outputs. `cancel` stops it.

In [ ]:
#| export
SAGEMAKER_STATUS = {"InProgress": "running", "Stopping": "running", "Completed": "complete", "Failed": "error", "Stopped": "stopped"}
LOG_GROUP = "/aws/sagemaker/TrainingJobs"

def _job_spec(b:SageMaker, job:Job) -> dict:
    "The job's spec, made ready for SageMaker: no push from the container (no Hub token there), outputs in the model on an emulator"
    folder = Path(job.folder)
    spec = json.loads((folder / "job.json").read_text())
    spec["push"], spec["outputs_in_model"] = None, emulated(b)
    (folder / "job.json").write_text(json.dumps(spec, indent=1))
    return spec

def upload_job(b:SageMaker, job:Job) -> tuple:
    "Upload a job's folder (the `code` channel) and its local data (channels of their own); returns the channels, spec, bucket and prefix"
    spec = _job_spec(b, job)
    bk, prefix, s3 = bucket(b), f"sysone/jobs/{job.id}", aws_client(b, "s3")
    s3.put_object(Bucket=bk, Key=f"{prefix}/code/job.tar.gz", Body=pack(Path(job.folder), skip=()))
    channels = {"code": f"s3://{bk}/{prefix}/code/"}
    for ch, path in (spec.get("upload") or {}).items(): channels[ch] = s3_put(s3, bk, f"{prefix}/{ch}", Path(path))
    return channels | (spec.get("channels") or {}), spec, bk, prefix

@submit.register
def submit_sagemaker(b:SageMaker, job:Job) -> Job:
    channels, spec, bk, prefix = upload_job(b, job)
    req = training_job_request(b, job, spec, channels, bk, prefix)
    aws_client(b, "sagemaker").create_training_job(**req)
    job.ref, job.status = req["TrainingJobName"], "running"
    return job

def _log_text(b:SageMaker, job:Job) -> str:
    "The CloudWatch log lines of a job since the last call (the read positions kept in the job's folder)"
    logs, state_f = aws_client(b, "logs"), Path(job.folder) / "cloudwatch.json"
    tokens = json.loads(state_f.read_text()) if state_f.exists() else {}
    try: streams = logs.describe_log_streams(logGroupName=LOG_GROUP, logStreamNamePrefix=job.ref)["logStreams"]
    except Exception: return ""
    out = []
    for s in streams:
        kw = {"nextToken": tokens[s["logStreamName"]]} if s["logStreamName"] in tokens else {}
        r = logs.get_log_events(logGroupName=LOG_GROUP, logStreamName=s["logStreamName"], startFromHead=True, **kw)
        out += [e["message"] for e in r["events"]]
        tokens[s["logStreamName"]] = r["nextForwardToken"]
    state_f.write_text(json.dumps(tokens))
    return "".join(m if m.endswith("\n") else m + "\n" for m in out)

def _download(b:SageMaker, job:Job, desc:dict):
    "Bring model.tar.gz and the outputs of an ended job into its folder"
    s3, folder = aws_client(b, "s3"), Path(job.folder)
    uri = (desc.get("ModelArtifacts") or {}).get("S3ModelArtifacts")
    if not uri: return
    bk, key = uri[len("s3://"):].split("/", 1)
    s3.download_file(bk, key, str(folder / "model.tar.gz"))
    model = folder / "sm_model"
    with tarfile.open(folder / "model.tar.gz") as t: t.extractall(model, filter="data")
    try:                                                    # SageMaker's output.tar.gz holds the job's outputs.tar.gz
        s3.download_file(bk, key.rsplit("/", 1)[0] + "/output.tar.gz", str(folder / "output.tar.gz"))
        with tarfile.open(folder / "output.tar.gz") as t: t.extract("outputs.tar.gz", folder, filter="data")
    except Exception:
        if (model / "sysone_outputs.tar.gz").exists(): shutil.move(model / "sysone_outputs.tar.gz", folder / "outputs.tar.gz")

@poll.register
def poll_sagemaker(b:SageMaker, job:Job) -> str:
    desc = aws_client(b, "sagemaker").describe_training_job(TrainingJobName=job.ref)
    text = _log_text(b, job)
    with open(Path(job.folder) / "job.log", "a") as f: f.write(text)
    st = SAGEMAKER_STATUS.get(desc["TrainingJobStatus"], "running")
    if st in DONE:
        _download(b, job, desc)
        tb = Path(job.folder) / "outputs.tar.gz"
        if tb.exists():
            with tarfile.open(tb) as t: st = json.load(t.extractfile("outputs/status.json")).get("status", st)
        job.status = st
        if st == "error" and desc.get("FailureReason"): job.last = (job.last or {}) | {"event": "failure", "reason": desc["FailureReason"][:300]}
    return text

@cancel.register
def cancel_sagemaker(b:SageMaker, job:Job) -> Job:
    aws_client(b, "sagemaker").stop_training_job(TrainingJobName=job.ref)
    job.status = "stopped"
    return job

@collect.register
def collect_sagemaker(b:SageMaker, job:Job, dest:Path) -> Path:
    folder = Path(job.folder)
    if (folder / "outputs.tar.gz").exists(): out = unpack_outputs(folder / "outputs.tar.gz", dest)
    elif job.done: out = Path(dest) / "outputs"; out.mkdir(parents=True, exist_ok=True)
    else: raise RuntimeError(f"{job.id} is {job.status}: its outputs come when it ends (job.wait())")
    model = folder / "sm_model"
    if not (out / "export" / "sysone.json").exists() and (model / "sysone.json").exists():
        shutil.copytree(model, out / "export", dirs_exist_ok=True, ignore=shutil.ignore_patterns("sysone_outputs.tar.gz"))
    return out

## With the SageMaker Python SDK

Teams that launch training with the SageMaker Python SDK can launch a sysone job with it too: `model_trainer(job)` returns the SDK's `ModelTrainer` for a job prepared with `remote(..., on=SageMaker(...), dry_run=True)`, with the same image, channels and time limit, so `.train()` runs it as `submit` would. With `SageMaker(local=True)` it is the SDK's local mode: the channels are local folders, and Docker runs the image here, with no AWS account at all. The SDK is not a dependency of sysone (`pip install sagemaker`; it pulls in heavy packages of its own).

::: {.callout-note title="Finding: the SageMaker Python SDK as an extra downgraded two packages for every install"}
`uv` locks one version of each package for all extras together. With `sagemaker` 3.23 as an extra, the lock moved `rich` from 15.0.0 to 14.3.4 and `importlib-metadata` from 9.0.1 to 6.11.0 for everyone, extra or not, to fit the SDK's own pins. So sysone's AWS extra is boto3 and the MLflow plugin only, and the SDK is installed beside sysone when wanted.
:::

In [ ]:
#| export
def local_channels(job:Job) -> dict:
    "A job's channels as local folders, for the SDK's local mode: the job's folder packed as `code`, and its local data"
    spec = _job_spec(SageMaker(local=True), job)
    root = Path(job.folder).parent / f"{job.id}-channels"
    (root / "code").mkdir(parents=True, exist_ok=True)
    (root / "code" / "job.tar.gz").write_bytes(pack(Path(job.folder), skip=()))
    channels = {"code": str(root / "code")}
    for ch, path in (spec.get("upload") or {}).items():
        d = root / ch; d.mkdir(exist_ok=True)
        (shutil.copytree(path, d, dirs_exist_ok=True) if Path(path).is_dir() else shutil.copy2(path, d))
        channels[ch] = str(d)
    return channels | (spec.get("channels") or {})

def model_trainer(job:Job, b:SageMaker|None=None, **kw):
    "The SageMaker Python SDK's ModelTrainer for a sysone job made with `remote(..., on=SageMaker(...), dry_run=True)`; local mode with `SageMaker(local=True)`"
    try:
        from sagemaker.train.model_trainer import ModelTrainer, Mode
        from sagemaker.train.configs import Compute, InputData, StoppingCondition
    except ImportError: raise ImportError("the SageMaker Python SDK is not installed: pip install sagemaker") from None
    b = b or job.backend_obj
    channels = local_channels(job) if b.local else upload_job(b, job)[0]
    spec = json.loads((Path(job.folder) / "job.json").read_text())
    args = dict(training_image=image_for(b, job.machine), compute=Compute(instance_type=job.machine, instance_count=1),
                input_data_config=[InputData(channel_name=ch, data_source=src) for ch, src in channels.items()],
                stopping_condition=StoppingCondition(max_runtime_in_seconds=int(spec.get("max_seconds") or 24 * 3600)),
                base_job_name=re.sub(r"[^A-Za-z0-9-]", "-", job.id)[:40])
    if b.local:
        root = Path(job.folder) / "sdk-local"
        for d in ("model", "algo-1/output/data", "output/data"): (root / d).mkdir(parents=True, exist_ok=True)   # see the finding below
        args |= {"training_mode": Mode.LOCAL_CONTAINER, "local_container_root": str(root)}
    else: args |= {"role": role_arn(b)}
    return ModelTrainer(**(args | kw))

::: {.callout-note title="Finding: on Docker Desktop, the SDK's local mode lost the model the job wrote"}
The SDK's local mode (sagemaker 3.23) mounts `<root>/model` as `/opt/ml/model` without making the folder first. On Linux, Docker makes a missing bind-mount folder on the host; Docker Desktop on a Mac makes it inside its own VM, so the container wrote the export there, the SDK found nothing to collect, and `model.tar.gz` came back empty while the job reported success. `model_trainer` makes the folders before the SDK runs, and the export comes back.
:::

In [ ]:
#| sagemaker
for k, v in {"AWS_ACCESS_KEY_ID": "test", "AWS_SECRET_ACCESS_KEY": "test", "AWS_DEFAULT_REGION": "us-east-1"}.items(): os.environ.setdefault(k, v)
os.environ |= {"SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}
sdk_script = Path(tempfile.mkdtemp()) / "train.py"
sdk_script.write_text("""from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, str(job_input("encoder")), train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
learn.export(job_output())
""")
lj = remote(sdk_script, on=SageMaker(local=True), hw="cpu", data={"data": "fixtures/typed_decisions_tiny.jsonl", "encoder": "fixtures/tiny-modernbert"},
            dry_run=True)
model_trainer(lj).train(wait=True, logs=False)
sdk_out = Path(tempfile.mkdtemp())
with tarfile.open(Path(lj.folder) / "sdk-local" / "compressed_artifacts" / "model.tar.gz") as t: t.extractall(sdk_out, filter="data")
rec0 = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(set(Decider.load(sdk_out, device="cpu").predict(rec0["state"], rec0["questions"])), set(rec0["questions"]))

## On floci

These cells need Docker; they are flagged `floci`, so the tests skip them unless run with `--flags floci`. They start the emulator, build sysone's CPU image (once: about 300 MB of downloads, PyTorch's CPU wheel the largest), and run the tiny training as a training job: the job's folder, the data and the tiny encoder go up as channels, the job runs in a container on this machine, its progress lines come through CloudWatch, and the export comes back from `model.tar.gz`.

In [ ]:
#| floci
url = floci_up()
tag = build_image()
script = Path(tempfile.mkdtemp()) / "train.py"
script.write_text("""from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, str(job_input("encoder")), train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
learn.export(job_output())
""")
os.environ |= {"SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}
sj = remote(script, on=SageMaker(endpoint_url=url), hw="cpu",
            data={"data": "fixtures/typed_decisions_tiny.jsonl", "encoder": "fixtures/tiny-modernbert"})
sj.wait(every=3)
test_eq(sj.status, "complete")

In [ ]:
#| floci
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
exp = sj.fetch()
test_eq(set(Decider.load(exp, device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))
assert [r for r in sj.progress() if r["event"] == "log"]                              # through CloudWatch
meta = json.loads((exp / "sysone.json").read_text())
test_eq(meta["training"]["hardware"]["run"]["backend"], "sagemaker")
jobs(backend="sagemaker")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()